# Weather Forecasting, Clustering, and Anomaly Detection

This notebook uses the canonical Open-Meteo hourly table from the Gomel dataset to demonstrate three scikit-learn workflows:

- hourly temperature forecasting as the main supervised task
- weather-regime clustering on daily temperature profiles
- anomaly detection on a compact set of weather features

The notebook reuses the project loader in `src/open_meteo.py` so the parsing logic stays consistent with the CLI.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.cluster import (
    DBSCAN,
    AgglomerativeClustering,
    Birch,
    KMeans,
    MiniBatchKMeans,
    SpectralClustering,
)
from sklearn.covariance import EllipticEnvelope
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.inspection import permutation_importance
from sklearn.linear_model import (
    BayesianRidge,
    ElasticNet,
    HuberRegressor,
    Lasso,
    LinearRegression,
    QuantileRegressor,
    Ridge,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsRegressor, LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, OneClassSVM

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## Load the canonical hourly table

The helper below converts the project `HourlyTable` object into NumPy arrays plus lightweight calendar features.

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {
    name: np.asarray(values, dtype=float) for name, values in table.columns.items()
}
timestamps = np.asarray(table.timestamps)

required_columns = [
    "misc__temperature_2m",
    "misc__relative_humidity_2m",
    "misc__dew_point_2m",
    "misc__apparent_temperature",
    "misc__pressure_msl",
    "misc__precipitation",
    "misc__vapour_pressure_deficit",
    "wind__wind_speed_10m",
    "wind__wind_gusts_10m",
    "cloud_cover__cloud_cover",
    "solar_radiation_vars__shortwave_radiation",
    "additional_vars_and_options__is_day",
    "european_air_quality_index__european_aqi",
]

missing = [name for name in required_columns if name not in column_arrays]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")


def build_calendar_features(time_values: np.ndarray) -> dict[str, np.ndarray]:
    hours = np.asarray([int(value[11:13]) for value in time_values], dtype=float)
    months = np.asarray([int(value[5:7]) for value in time_values], dtype=float)
    days = np.asarray([int(value[8:10]) for value in time_values], dtype=float)
    day_of_year = np.asarray(
        [
            int(
                (np.datetime64(value[:10]) - np.datetime64(value[:4] + "-01-01"))
                / np.timedelta64(1, "D")
            )
            + 1
            for value in time_values
        ],
        dtype=float,
    )
    return {
        "hour": hours,
        "month": months,
        "day": days,
        "day_of_year": day_of_year,
        "hour_sin": np.sin(2.0 * np.pi * hours / 24.0),
        "hour_cos": np.cos(2.0 * np.pi * hours / 24.0),
        "day_of_year_sin": np.sin(2.0 * np.pi * day_of_year / 366.0),
        "day_of_year_cos": np.cos(2.0 * np.pi * day_of_year / 366.0),
    }


calendar_features = build_calendar_features(timestamps)

print(f"Rows: {len(timestamps):,}")
print(f"Columns: {len(column_arrays):,}")
print(f"Start: {timestamps[0]}")
print(f"End:   {timestamps[-1]}")

## Build a supervised forecasting dataset

We forecast the next hour of `temperature_2m` using current weather signals plus lagged temperature terms.

In [ ]:
target_name = "misc__temperature_2m"
base_feature_names = [
    "misc__temperature_2m",
    "misc__relative_humidity_2m",
    "misc__dew_point_2m",
    "misc__apparent_temperature",
    "misc__pressure_msl",
    "misc__precipitation",
    "misc__vapour_pressure_deficit",
    "wind__wind_speed_10m",
    "wind__wind_gusts_10m",
    "cloud_cover__cloud_cover",
    "solar_radiation_vars__shortwave_radiation",
    "additional_vars_and_options__is_day",
    "european_air_quality_index__european_aqi",
]
lag_hours = [1, 3, 6, 24]
forecast_horizon = 1
max_lag = max(lag_hours)
row_start = max_lag
row_end = len(timestamps) - forecast_horizon
rows = row_end - row_start

feature_blocks = []
feature_names = []

for name in base_feature_names:
    feature_blocks.append(column_arrays[name][row_start:row_end])
    feature_names.append(name)

for name, values in calendar_features.items():
    feature_blocks.append(values[row_start:row_end])
    feature_names.append(name)

target_values = column_arrays[target_name]
for lag in lag_hours:
    feature_blocks.append(target_values[row_start - lag : row_end - lag])
    feature_names.append(f"{target_name}__lag_{lag}h")

feature_matrix = np.column_stack(feature_blocks)
target = target_values[row_start + forecast_horizon : row_end + forecast_horizon]
sample_timestamps = timestamps[row_start:row_end]

valid_mask = np.isfinite(feature_matrix).all(axis=1) & np.isfinite(target)
X = feature_matrix[valid_mask]
y = target[valid_mask]
time_index = sample_timestamps[valid_mask]

print(f"Supervised rows: {len(y):,}")
print(f"Feature count: {X.shape[1]}")
print("First features:", feature_names[:8])

In [ ]:
def chronological_split(
    X: np.ndarray,
    y: np.ndarray,
    time_index: np.ndarray,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.15,
):
    total = len(y)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        "X_train": X[:train_end],
        "y_train": y[:train_end],
        "t_train": time_index[:train_end],
        "X_valid": X[train_end:valid_end],
        "y_valid": y[train_end:valid_end],
        "t_valid": time_index[train_end:valid_end],
        "X_test": X[valid_end:],
        "y_test": y[valid_end:],
        "t_test": time_index[valid_end:],
    }


split = chronological_split(X, y, time_index)
for name in ["train", "valid", "test"]:
    print(name, split[f"X_{name}"].shape, split[f"t_{name}"][0], split[f"t_{name}"][-1])

## Train regression baselines

We compare linear, latent-factor, distance-based, boosting, and tree-based regressors on the same chronological split. SVR and Gaussian Process regression are trained on downsampled subsets to keep notebook runtime practical.

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
    }


models = {
    "linear_regression": Pipeline(
        [("scale", StandardScaler()), ("model", LinearRegression())]
    ),
    "ridge": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "lasso": Pipeline(
        [("scale", StandardScaler()), ("model", Lasso(alpha=0.001, max_iter=5000))]
    ),
    "bayesian_ridge": Pipeline(
        [("scale", StandardScaler()), ("model", BayesianRidge())]
    ),
    "huber": Pipeline([("scale", StandardScaler()), ("model", HuberRegressor())]),
    "quantile_regressor": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")),
        ]
    ),
    "elastic_net": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000)),
        ]
    ),
    "pls_regression": Pipeline(
        [("scale", StandardScaler()), ("model", PLSRegression(n_components=8))]
    ),
    "random_forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        learning_rate=0.08,
        max_depth=10,
        max_iter=200,
        random_state=42,
    ),
    "adaboost_regressor": AdaBoostRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
    ),
    "bagging_regressor": BaggingRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "knn_regressor": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=12, weights="distance")),
        ]
    ),
}

results = []
predictions = {}

for model_name, model in models.items():
    model.fit(split["X_train"], split["y_train"])
    valid_pred = model.predict(split["X_valid"])
    test_pred = model.predict(split["X_test"])
    valid_metrics = regression_metrics(split["y_valid"], valid_pred)
    test_metrics = regression_metrics(split["y_test"], test_pred)
    results.append(
        {
            "model": model_name,
            "valid_mae": valid_metrics["mae"],
            "valid_rmse": valid_metrics["rmse"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
        }
    )
    predictions[model_name] = test_pred

svr_train_limit = min(8000, len(split["y_train"]))
svr = Pipeline(
    [
        ("scale", StandardScaler()),
        ("model", SVR(C=5.0, epsilon=0.1, gamma="scale")),
    ]
)
svr.fit(split["X_train"][:svr_train_limit], split["y_train"][:svr_train_limit])
svr_test_pred = svr.predict(split["X_test"][:2000])
svr_metrics = regression_metrics(split["y_test"][:2000], svr_test_pred)
results.append(
    {
        "model": "svr_subset",
        "valid_mae": math.nan,
        "valid_rmse": math.nan,
        "test_mae": svr_metrics["mae"],
        "test_rmse": svr_metrics["rmse"],
        "test_r2": svr_metrics["r2"],
    }
)
predictions["svr_subset"] = svr_test_pred

gpr_train_limit = min(2500, len(split["y_train"]))
gpr_eval_limit = min(1500, len(split["y_test"]), len(split["y_valid"]))
gpr = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            GaussianProcessRegressor(
                kernel=RBF(length_scale=2.0) + WhiteKernel(noise_level=1.0),
                alpha=1e-3,
                normalize_y=True,
                random_state=42,
            ),
        ),
    ]
)
gpr.fit(split["X_train"][:gpr_train_limit], split["y_train"][:gpr_train_limit])
gpr_valid_pred = gpr.predict(split["X_valid"][:gpr_eval_limit])
gpr_test_pred = gpr.predict(split["X_test"][:gpr_eval_limit])
gpr_valid_metrics = regression_metrics(
    split["y_valid"][:gpr_eval_limit], gpr_valid_pred
)
gpr_test_metrics = regression_metrics(split["y_test"][:gpr_eval_limit], gpr_test_pred)
results.append(
    {
        "model": "gaussian_process_subset",
        "valid_mae": gpr_valid_metrics["mae"],
        "valid_rmse": gpr_valid_metrics["rmse"],
        "test_mae": gpr_test_metrics["mae"],
        "test_rmse": gpr_test_metrics["rmse"],
        "test_r2": gpr_test_metrics["r2"],
    }
)
predictions["gaussian_process_subset"] = gpr_test_pred

for row in sorted(results, key=lambda item: item["test_rmse"]):
    print(row)

In [ ]:
best_model_name = min(
    (
        row
        for row in results
        if row["model"] not in {"svr_subset", "gaussian_process_subset"}
    ),
    key=lambda item: item["test_rmse"],
)["model"]
best_model = models[best_model_name]
importance = permutation_importance(
    best_model,
    split["X_test"][:4000],
    split["y_test"][:4000],
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)
top_indices = np.argsort(importance.importances_mean)[-10:][::-1]
for index in top_indices:
    print(feature_names[index], float(importance.importances_mean[index]))

In [ ]:
plot_count = 7 * 24
fig, ax = plt.subplots()
ax.plot(
    split["t_test"][:plot_count],
    split["y_test"][:plot_count],
    label="actual",
    linewidth=2,
)
ax.plot(
    split["t_test"][:plot_count],
    predictions[best_model_name][:plot_count],
    label=best_model_name,
    linewidth=1.8,
)
ax.set_title("Next-hour temperature forecast on the first week of the test split")
ax.set_ylabel("Temperature (C)")
ax.set_xlabel("Timestamp")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

## Cluster daily temperature regimes

A simple daily 24-hour temperature profile gives an interpretable representation for centroid-based, hierarchical, graph-based, density-based, Birch, and mixture-model clustering.

In [ ]:
temperature = column_arrays["misc__temperature_2m"]
full_days = len(temperature) // 24
daily_profiles = temperature[: full_days * 24].reshape(full_days, 24)
daily_mean = daily_profiles.mean(axis=1, keepdims=True)
daily_std = daily_profiles.std(axis=1, keepdims=True)
daily_features = np.concatenate([daily_profiles, daily_mean, daily_std], axis=1)

cluster_models = {
    "kmeans": KMeans(n_clusters=4, n_init=20, random_state=42),
    "mini_batch_kmeans": MiniBatchKMeans(
        n_clusters=4, batch_size=256, n_init=20, random_state=42
    ),
    "birch": Birch(n_clusters=4, threshold=1.5),
    "agglomerative": AgglomerativeClustering(n_clusters=4),
    "dbscan": DBSCAN(eps=12.0, min_samples=10),
    "spectral_subset": SpectralClustering(
        n_clusters=4,
        affinity="nearest_neighbors",
        assign_labels="kmeans",
        random_state=42,
    ),
    "gaussian_mixture": GaussianMixture(
        n_components=4, covariance_type="full", random_state=42
    ),
}
cluster_assignments = {}
for model_name, cluster_model in cluster_models.items():
    if model_name == "spectral_subset":
        spectral_sample = min(1200, len(daily_features))
        labels = cluster_model.fit_predict(daily_features[:spectral_sample])
        cluster_assignments[model_name] = labels
    elif model_name == "gaussian_mixture":
        labels = cluster_model.fit_predict(daily_features)
        cluster_assignments[model_name] = labels
    else:
        labels = cluster_model.fit_predict(daily_features)
        cluster_assignments[model_name] = labels
    unique_labels, counts = np.unique(labels, return_counts=True)
    print(model_name)
    for label, count in zip(unique_labels, counts):
        print(f"  cluster {label}: {count} days")
    if model_name == "dbscan" and -1 in unique_labels:
        noise_count = counts[np.where(unique_labels == -1)[0][0]]
        print(f"  noise points: {noise_count} days")
cluster_labels = cluster_assignments["kmeans"]

In [ ]:
fig, axes = plt.subplots(
    1, len(cluster_assignments), figsize=(6 * len(cluster_assignments), 5), sharey=True
)
axes = np.atleast_1d(axes)
hours = np.arange(24)
for ax, (model_name, labels) in zip(axes, cluster_assignments.items()):
    if model_name == "spectral_subset":
        source_profiles = daily_profiles[: len(labels)]
    else:
        source_profiles = daily_profiles
    for label in np.unique(labels):
        centroid = source_profiles[labels == label].mean(axis=0)
        ax.plot(hours, centroid, label=f"cluster {label}")
    ax.set_title(f"Mean 24-hour profile by {model_name}")
    ax.set_xlabel("Hour of day")
    ax.legend()
axes[0].set_ylabel("Temperature (C)")
plt.tight_layout()

## Detect anomalies with Isolation Forest, Local Outlier Factor, One-Class SVM, and Elliptic Envelope

This section uses a compact multivariate slice and a modest sample size to highlight unusual hours without turning the notebook into a long-running batch job. Isolation Forest, Local Outlier Factor, One-Class SVM, and Elliptic Envelope are compared side by side.

In [ ]:
anomaly_feature_names = [
    "misc__temperature_2m",
    "misc__pressure_msl",
    "misc__precipitation",
    "wind__wind_speed_10m",
    "wind__wind_gusts_10m",
    "cloud_cover__cloud_cover",
    "solar_radiation_vars__shortwave_radiation",
    "european_air_quality_index__european_aqi",
]
anomaly_matrix = np.column_stack(
    [column_arrays[name] for name in anomaly_feature_names]
)
anomaly_mask = np.isfinite(anomaly_matrix).all(axis=1)
anomaly_matrix = anomaly_matrix[anomaly_mask]
anomaly_times = timestamps[anomaly_mask]

anomaly_sample = min(12000, len(anomaly_matrix))
anomaly_matrix = anomaly_matrix[-anomaly_sample:]
anomaly_times = anomaly_times[-anomaly_sample:]

scaled_anomaly_matrix = StandardScaler().fit_transform(anomaly_matrix)
anomaly_models = {
    "isolation_forest": IsolationForest(
        n_estimators=200, contamination=0.01, random_state=42
    ),
    "local_outlier_factor": LocalOutlierFactor(n_neighbors=35, contamination=0.01),
    "one_class_svm": OneClassSVM(kernel="rbf", gamma="scale", nu=0.01),
    "elliptic_envelope": EllipticEnvelope(contamination=0.01, random_state=42),
}
anomaly_results = {}
for model_name, anomaly_model in anomaly_models.items():
    if model_name == "local_outlier_factor":
        labels = anomaly_model.fit_predict(scaled_anomaly_matrix)
        scores = anomaly_model.negative_outlier_factor_
    else:
        anomaly_model.fit(scaled_anomaly_matrix)
        labels = anomaly_model.predict(scaled_anomaly_matrix)
        scores = anomaly_model.score_samples(scaled_anomaly_matrix)
    anomaly_order = np.argsort(scores)[:10]
    anomaly_results[model_name] = {
        "labels": labels,
        "scores": scores,
        "order": anomaly_order,
    }
    print(model_name)
    for index in anomaly_order:
        print(f"  {anomaly_times[index]} {float(scores[index])}")

In [ ]:
fig, axes = plt.subplots(
    1, len(anomaly_results), figsize=(7 * len(anomaly_results), 5), sharey=True
)
axes = np.atleast_1d(axes)
for ax, (model_name, result) in zip(axes, anomaly_results.items()):
    anomaly_order = result["order"]
    ax.plot(anomaly_times, anomaly_matrix[:, 0], label="temperature_2m", alpha=0.8)
    ax.scatter(
        anomaly_times[anomaly_order],
        anomaly_matrix[anomaly_order, 0],
        color="crimson",
        label="flagged anomalies",
        zorder=3,
    )
    ax.set_title(f"{model_name} anomalies on the recent sample window")
    ax.set_xlabel("Timestamp")
    ax.legend()
axes[0].set_ylabel("Temperature (C)")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

## Suggested follow-up experiments

- switch the target from `temperature_2m` to `european_aqi` or `wind_speed_10m`
- add rolling-window and derivative features from the canonical hourly table
- aggregate hourly weather to daily features and connect them to the flood dataset
- compare these baselines with sequence models later, but keep the chronological split and leakage checks unchanged